# 1. Sen4AgriNet data and satellite imagery

Prepare crop-reference locations and retrieve matching Sentinel-2 imagery. Sen4AgriNet supplies the current carrot/potato pilot. FranceCrops cabbage has its own [preparation notebook](02_francecrops_preparation.ipynb).

**Run order:** this notebook → [Four-month windows](03_four_month_windows.ipynb) → [Model training](05_model_training.ipynb).

Satellite imagery is retrieved for the reference locations and dates; crop labels are not converted into imagery.

**Pilot budget:** collect 3,400 carrot and 10,000 potato candidate pixels, then retain up to 500 complete usable pixels per crop in the window notebook. Expanded bilinear acquisition files use separate `bilinear_expanded` folders; previous outputs are preserved.

Existing code below retains its original section numbers and output explanations.



**Expanded timing-first bilinear run:** 3,400 carrot and 10,000 potato candidate pixels. Daily summaries, satellite downloads, and Earth Engine sign-in are enabled for this rerun. Run sequentially: offline capacity and sampling → daily summaries → timing gate → qualified-window imagery → handoff. Existing pilot files remain in their original folders.

**Minimum acquisition pool:** each crop must have more than 3,000 unique sampled pixels remaining after timing filtering before imagery downloading starts. Repeated half-month images and rectangular context pixels do not increase this count. The final usable target remains 500 per crop; incomplete pixels remain excluded.


## Start here: choose the run mode

Run sequentially with `download_netcdf_files = False`, `rebuild_field_inventory = False`, and `install_dependencies = False` to reuse saved references. No NetCDF downloads are needed. New outputs use `bilinear_expanded` folders, preserving existing pilot files.

Both `retrieve_daily_summaries` and `download_satellite_imagery` are set to `True` for this rerun. Offline capacity checks happen first; the current carrot budget is 3,400, within its 3,471 capped-pixel capacity. Add reference fields if a later capacity or timing-stage check reports a shortfall. Budgets and the 100-pixel field cap are not reduced.

Daily summaries and imagery are already enabled for this rerun. The timing gate runs before any imagery transfer. Only timing-qualified eight-bin windows reach the downloader. Adjacent-year needs remain pending.


In [ ]:
from pathlib import Path
import json
import os

# Reuse saved field references by default; satellite retrieval still runs below.
download_netcdf_files = False
rebuild_field_inventory = False
install_dependencies = False

# Collection is enabled for this rerun; successful downloads remain reusable.
from crop_pipeline.imagery import bilinear as bilinear_collection
run_name = bilinear_collection.COLLECTION_NAME
retrieve_daily_summaries = True
download_satellite_imagery = True

# Notebooks live in notebooks/; run from the repository root so relative paths resolve.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
settings_path = Path("settings.local.json")
local_settings = (
    json.loads(settings_path.read_text(encoding="utf-8")) if settings_path.exists() else {}
)

# 1. Choose where the dataset is stored.
# This folder must contain the "data" folder.
# Replace this example with your own storage location.
dataset_folder = (
    Path(os.environ.get("S4A_ROOT", local_settings.get("dataset_root", "./S4A")))
    .expanduser()
    .resolve()
)

# 2. Choose where small inventory reports will be saved.
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).resolve()

# 3. Select ONE tile and ONE batch.
dataset_name = "orion-ai-lab/S4A"
year = 2020  # Keep 2019 for future work.
tile = "31TBF"
files_per_batch = 50
batch_number = 1  # Batch numbers start at 1.

if year != 2020:
    raise ValueError("Step 2 currently processes 2020 only.")
if files_per_batch < 1 or batch_number < 1:
    raise ValueError("Batch size and batch number must be at least 1.")

print("Dataset folder:", dataset_folder)
print("Selected tile:", tile)
print("Selected batch:", batch_number)

field_folder = report_folder.parent / "fields" / "2020"
saved_candidate_path = field_folder / "download_candidates.geojson"
if not rebuild_field_inventory and not saved_candidate_path.is_file():
    raise FileNotFoundError(
        f"Saved field inventory is missing: {saved_candidate_path}. "
        "Restore sen4agrinet_work/fields/2020, or enable rebuilding when raw NetCDFs are available."
    )
print("NetCDF downloading enabled:", download_netcdf_files)
print("Rebuild fields from NetCDF:", rebuild_field_inventory)
print("Saved field inventory:", saved_candidate_path)


### Environment packages — optional

Output: `Dependency installation skipped` — the current kernel environment is reused. Set `install_dependencies = True` only when installing the pinned requirements is necessary.


In [ ]:
if install_dependencies:
    import subprocess
    import sys

    # Use the active kernel's interpreter so packages reach this notebook's environment.
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-r", "requirements-notebooks.txt"]
    )
else:
    print("Dependency installation skipped: using the current kernel environment.")


## 1. Define the study

The study asks whether external crop-reference data can improve Random Forest classification in Benguet when verified local labels are limited.

| Setting | Plan |
|---|---|
| Crops | Cabbage, carrot and potato |
| Target area | Benguet; exact boundary and map reference period still need agreement |
| Imagery | Sentinel-2 L2A surface reflectance |
| Classifier window | Four consecutive months, with eight half-month intervals |
| Study bands | B3, B4, B5, B8 and B11 |
| Indices | NDVI, NDRE and NDMI |
| Feature comparison | 72 interval features, 120 interval-plus-shape features, and 48 shape-only features in the feature notebook |
| Training comparison | External-only, local-only, and external plus the same local fields |
| Evaluation | The same independent Benguet test fields for all models |

The expanded development pilot collects 3,000 carrot candidates and 3,000 potato candidates, then keeps up to 500 usable pixels per crop after filtering. It is not a complete crop inventory. Pixels from the same parcel group must stay together in later splits.

The proposed window starts in the month containing accepted green-up. This rule still needs validation. Annual crop declarations do not prove which crop grew during a particular cycle. Confirm crop-cycle relevance before accepting training seasons.

Raw data and imagery stay on the configured data drive. Small reports live under `sen4agrinet_work`. Keep local paths in `settings.local.json`; that file is excluded from Git.


### Outputs and interpretation

A shared study scope and folder plan. Exact study boundaries, local label budgets, map dates and validation decisions remain open.


## 2. Optional NetCDF preparation — disabled by default

This raw-data section is at the beginning so the rest of the notebook can run sequentially. With `download_netcdf_files = False`, it does not contact Hugging Face or download NetCDFs.

Output: `NetCDF retrieval skipped: download_netcdf_files=False.` — intentional reuse mode, not an error. Continue running the next cell.

To start from scratch, enable downloading for the desired batch and enable field rebuilding. One batch does not guarantee enough labeled fields for the full pilot.


### 2.3 Preview one batch — optional

Save/reuse the remote file list and display the selected batch. This requests metadata, not imagery.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Batch: <number> of <number>` | The selected batch and total available batches for this tile. Use this to track raw-data retrieval. |
| `Files in this batch: <number>` | How many raw NetCDF files belong to this batch; not how many were newly downloaded. |
| `Full batch size (GiB): <value>` | The total expected size of the batch, including files already present. It estimates storage, not remaining transfer size. |
| `Dataset version: <revision>` | The pinned source revision used to reproduce the file list and downloads. |
| `Already present: <filename>` | A local file has the expected size and can be reused by the download cell. This preview has not checked its header or crop labels. |
| `Needs download: <filename>` | The file is absent or its size differs from the saved remote metadata. Only the next optional cell transfers it. |
<!-- output-guide:end -->


In [ ]:
if download_netcdf_files:
    import json
    from huggingface_hub import HfApi

    # Save one file list for each tile so batch membership stays consistent.
    tile_report_folder = report_folder / str(year) / tile
    tile_report_folder.mkdir(parents=True, exist_ok=True)
    file_list_path = tile_report_folder / "download_plan.json"

    if file_list_path.exists():
        # Reuse the saved file list and dataset version.
        download_plan = json.loads(file_list_path.read_text(encoding="utf-8"))
        if (
            download_plan["repo_id"] != dataset_name
            or download_plan["year"] != year
            or download_plan["tile"] != tile
        ):
            raise ValueError("The saved file list belongs to a different dataset or tile.")
        if download_plan["batch_size"] != files_per_batch:
            raise ValueError("Use the saved batch size or choose a different report folder.")
    else:
        # Request filenames and sizes only. This does not download the imagery.
        hub = HfApi()
        dataset_version = hub.repo_info(dataset_name, repo_type="dataset").sha
        remote_tile_folder = f"data/{year}/{tile}"
        remote_items = hub.list_repo_tree(
            repo_id=dataset_name,
            repo_type="dataset",
            revision=dataset_version,
            path_in_repo=remote_tile_folder,
            recursive=True,
        )

        available_files = []
        for item in remote_items:
            is_netcdf_file = item.path.endswith(".nc") and hasattr(item, "size")
            if is_netcdf_file:
                available_files.append({"filename": item.path, "expected_bytes": item.size})

        available_files.sort(key=lambda file: file["filename"])
        if not available_files:
            raise ValueError("No NetCDF files were found in the selected tile.")

        download_plan = {
            "repo_id": dataset_name,
            "revision": dataset_version,
            "year": year,
            "tile": tile,
            "batch_size": files_per_batch,
            "files": available_files,
        }
        file_list_path.write_text(json.dumps(download_plan, indent=2), encoding="utf-8")

    # Pick only the filenames belonging to the selected batch.
    first_file = (batch_number - 1) * files_per_batch
    last_file = first_file + files_per_batch
    batch_files = download_plan["files"][first_file:last_file]

    if not batch_files:
        raise ValueError("This batch number is beyond the end of the file list.")

    batch_name = f"s4a_{year}_{tile}_batch_{batch_number:03d}"
    number_of_batches = (len(download_plan["files"]) + files_per_batch - 1) // files_per_batch
    batch_bytes = sum(file["expected_bytes"] for file in batch_files)

    print("Batch:", batch_number, "of", number_of_batches)
    print("Files in this batch:", len(batch_files))
    print("Full batch size (GiB):", round(batch_bytes / 1024**3, 2))
    print("Dataset version:", download_plan["revision"])

    for file in batch_files:
        local_file = dataset_folder / file["filename"]
        if local_file.is_file() and local_file.stat().st_size == file["expected_bytes"]:
            print("Already present:", file["filename"])
        else:
            print("Needs download:", file["filename"])

else:
    print("NetCDF retrieval skipped: download_netcdf_files=False.")


### 2.4 Download one batch — optional

Retrieve missing or size-mismatched files, check headers and save progress after each file.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Downloading: <filename>` | A missing or size-mismatched NetCDF is being retrieved. Skip this optional cell when resuming without raw downloads. |
| `<filename> -> downloaded_header_readable` | The file was downloaded, its size matches and its header opens. Full array integrity and crop identity are not established. |
| `<filename> -> existing_header_readable` | An existing file passed the size/header checks and was reused. |
| `<filename> -> failed / Reason: <message>` | This file failed retrieval or verification. Its failure is recorded rather than interpreted as crop absence. |
| `Report saved to: <path>` | The batch CSV containing per-file status and errors; use it when resuming or investigating failures. |
| `Failed files: <number>` | Failed entries in this batch progress record. Zero means no recorded failures in that record, not that the full dataset is complete. |
<!-- output-guide:end -->


In [ ]:
if download_netcdf_files:
    import csv
    from datetime import datetime, timezone
    from huggingface_hub import hf_hub_download
    from netCDF4 import Dataset

    # This cell processes ONLY the batch shown in the previous cell.
    manifest_path = tile_report_folder / f"{batch_name}.csv"
    report_columns = [
        "batch",
        "dataset",
        "version",
        "year",
        "tile",
        "filename",
        "local_path",
        "expected_bytes",
        "actual_bytes",
        "checked_at_utc",
        "status",
        "error",
    ]

    # Preserve previous results if this batch was interrupted.
    results_by_filename = {}
    if manifest_path.exists():
        with manifest_path.open(newline="", encoding="utf-8") as report:
            for row in csv.DictReader(report):
                results_by_filename[row["filename"]] = row

    def save_progress():
        """Save results after every file without leaving a half-written report."""
        temporary_path = manifest_path.with_suffix(".csv.tmp")
        with temporary_path.open("w", newline="", encoding="utf-8") as report:
            writer = csv.DictWriter(report, fieldnames=report_columns)
            writer.writeheader()
            for file in batch_files:
                if file["filename"] in results_by_filename:
                    writer.writerow(results_by_filename[file["filename"]])
        temporary_path.replace(manifest_path)

    for file in batch_files:
        filename = file["filename"]
        local_file = dataset_folder / filename
        expected_size = file["expected_bytes"]

        result = {
            "batch": batch_name,
            "dataset": dataset_name,
            "version": download_plan["revision"],
            "year": year,
            "tile": tile,
            "filename": filename,
            "local_path": str(local_file),
            "expected_bytes": expected_size,
            "actual_bytes": "",
            "checked_at_utc": datetime.now(timezone.utc).isoformat(),
            "status": "pending",
            "error": "",
        }

        try:
            # Download missing files or replace files with the wrong size.
            needs_download = True
            if local_file.is_file():
                needs_download = local_file.stat().st_size != expected_size

            if needs_download:
                print("Downloading:", filename)
                hf_hub_download(
                    repo_id=dataset_name,
                    repo_type="dataset",
                    revision=download_plan["revision"],
                    filename=filename,
                    local_dir=dataset_folder,
                    force_download=local_file.exists(),
                )

            # Check size, then open the header without loading imagery arrays.
            result["actual_bytes"] = local_file.stat().st_size
            if result["actual_bytes"] != expected_size:
                raise ValueError("The local file size does not match the remote file size.")

            with Dataset(local_file, "r") as netcdf_file:
                # Access metadata only. The context manager closes the file.
                group_names = list(netcdf_file.groups)

            result["status"] = (
                "downloaded_header_readable" if needs_download else "existing_header_readable"
            )

        except Exception as error:
            # Record the problem and continue with the other files in this batch.
            result["status"] = "failed"
            result["error"] = f"{type(error).__name__}: {error}"

        results_by_filename[filename] = result
        save_progress()
        print(filename, "->", result["status"])
        if result["error"]:
            print("  Reason:", result["error"])

    failed_files = 0
    for result in results_by_filename.values():
        if result["status"] == "failed":
            failed_files += 1

    print("Report saved to:", manifest_path)
    print("Failed files:", failed_files)

else:
    print("NetCDF retrieval skipped: download_netcdf_files=False.")


### Outputs and interpretation

`download_plan.json` stores a tile’s file list and revision. A per-batch CSV records sizes, header checks and errors. Raw files stay under the configured `data/2020/<tile>/` folder. A readable header is not a complete integrity check.


## 3. Inspect the downloaded 2020 dataset

Find crop labels, parcel IDs, required bands and observation dates in every downloaded patch. Read masks and small date arrays; do not load the spectral time series.

Original S4A codes are cabbage 213, carrot 231 and potato 510. The report lists all observed codes, but cannot automatically prove that the chosen encoding is correct.

Files are read one at a time. Checkpoints reuse successful results only when the file and inspection settings match. Failed files and warning-bearing files are checked again. Set `repeat_checks = True` to repeat every check.

The scan covers local files, not guaranteed completeness against the remote release. Parcel counts are occurrences across files, not deduplicated independent fields.


**Sequential reuse mode:** the raw-file code in this section runs only when `rebuild_field_inventory = True`. With the default `False`, each code cell prints a skip message and continues; no raw NetCDF files are opened.


### 3.1 Choose inspection scope

Find every local 2020 NetCDF file and set the label codes and report folder.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Dataset year: <year>` | The year being inspected; this source-data scan currently handles 2020. |
| `Local files found: <number>` | Raw NetCDF files found under the configured year folder. It measures local scope, not completeness against the online release. |
| `Reports: <path>` | Where dataset-inspection summaries and per-file checkpoints will be stored. |
<!-- output-guide:end -->


In [ ]:
if rebuild_field_inventory:
    import numpy as np
    from netCDF4 import Dataset, num2date

    # Inspect ALL downloaded 2020 tiles, regardless of the Step 2 batch selection.
    inspection_year = 2020
    raw_year_folder = dataset_folder / "data" / str(inspection_year)
    if not raw_year_folder.is_dir():
        raise FileNotFoundError(f"Dataset folder does not exist: {raw_year_folder}")

    # Read files sequentially; the file list itself is small metadata.
    dataset_files = sorted(raw_year_folder.rglob("*.nc"))
    if not dataset_files:
        raise ValueError("No downloaded NetCDF files were found for 2020.")

    crop_codes = {"cabbage": 213, "carrot": 231, "potato": 510}
    required_bands = ["B03", "B04", "B05", "B08", "B11"]
    parcel_background_ids = [0, -1]
    repeat_checks = False

    inspection_folder = report_folder / str(inspection_year) / "dataset_inspection"
    inspection_folder.mkdir(parents=True, exist_ok=True)
    print("Dataset year:", inspection_year)
    print("Local files found:", len(dataset_files))
    print("Reports:", inspection_folder)

else:
    print("Raw-data inspection and field extraction skipped: rebuild_field_inventory=False.")


### 3.2 Define inspection helpers

These functions read masks, find nested variables and decode dates. Definitions do not run a dataset scan.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `No printed output` | Defines the inspection helpers for later cells. No raw files are inspected and no crop counts are produced yet. |
<!-- output-guide:end -->


In [ ]:
if rebuild_field_inventory:

    def collect_variables(group):
        """Find variables inside the root and all NetCDF subgroups."""
        variables = []
        for variable in group.variables.values():
            variables.append(variable)
        for child_group in group.groups.values():
            variables.extend(collect_variables(child_group))
        return variables

    def find_one_variable(variables, name):
        matches = [variable for variable in variables if variable.name.lower() == name.lower()]
        if len(matches) != 1:
            raise ValueError(f"Expected one {name} variable; found {len(matches)}.")
        return matches[0]

    def read_mask(variable):
        """Read a 2D integer mask while preserving invalid-pixel information."""
        masked_values = np.ma.asarray(variable[:])
        if masked_values.ndim != 2:
            raise ValueError(f"{variable.name} is not a 2D mask: {masked_values.shape}")
        values = np.asarray(masked_values.data)
        valid = ~np.ma.getmaskarray(masked_values)
        valid = valid & np.isfinite(values)
        if np.any(values[valid] != np.floor(values[valid])):
            raise ValueError(f"{variable.name} contains noninteger values.")
        return values, valid

    def inspect_dates(variable):
        """Decode dates without guessing units or counting duplicate dates twice."""
        units = getattr(variable, "units", "")
        if "since" not in units:
            raise ValueError("Time coordinate lacks CF 'since' units.")
        time_values = np.ma.asarray(variable[:]).compressed()
        decoded_dates = num2date(
            time_values, units, calendar=getattr(variable, "calendar", "standard")
        )
        unique_dates = sorted(set(decoded_dates))
        gaps = []
        for earlier, later in zip(unique_dates, unique_dates[1:]):
            gaps.append((later - earlier).total_seconds() / 86400)
        return {
            "first_date": str(unique_dates[0]) if unique_dates else None,
            "last_date": str(unique_dates[-1]) if unique_dates else None,
            "unique_observations": len(unique_dates),
            "largest_gap_days": max(gaps) if gaps else None,
            "dates": [str(date) for date in unique_dates],
        }

    def inspect_file(local_file):
        """Inspect one file, then close it before processing another."""
        details = {"status": "ok", "warnings": [], "crops": {}, "bands": {}, "times": []}
        with Dataset(local_file, "r") as dataset:
            variables = collect_variables(dataset)
            details["metadata"] = {name: str(dataset.getncattr(name)) for name in dataset.ncattrs()}
            label_variable = find_one_variable(variables, "labels")
            labels, valid_labels = read_mask(label_variable)
            details["label_shape"] = list(labels.shape)
            codes, counts = np.unique(labels[valid_labels], return_counts=True)
            details["observed_codes"] = {
                str(int(code)): int(count) for code, count in zip(codes, counts)
            }

            parcels = None
            valid_parcels = None
            try:
                parcels, valid_parcels = read_mask(find_one_variable(variables, "parcels"))
                if parcels.shape != labels.shape:
                    raise ValueError("Label and parcel masks have different shapes.")
                valid_parcels = valid_parcels & ~np.isin(parcels, parcel_background_ids)
            except Exception as error:
                parcels = None
                details["warnings"].append(f"Parcel mask: {error}")

            for crop, code in crop_codes.items():
                crop_pixels = valid_labels & (labels == code)
                crop_result = {
                    "code": code,
                    "pixels": int(crop_pixels.sum()),
                    "parcel_occurrences": None,
                    "parcel_ids": None,
                    "pixels_without_valid_parcel": None,
                }
                if parcels is not None:
                    parcel_ids = np.unique(parcels[crop_pixels & valid_parcels])
                    crop_result["parcel_ids"] = [int(parcel_id) for parcel_id in parcel_ids]
                    crop_result["parcel_occurrences"] = len(parcel_ids)
                    crop_result["pixels_without_valid_parcel"] = int(
                        (crop_pixels & ~valid_parcels).sum()
                    )
                details["crops"][crop] = crop_result

            # Check band shapes, never read the spectral arrays here.
            for band in required_bands:
                matches = [variable for variable in variables if variable.name.upper() == band]
                details["bands"][band] = [
                    {"shape": list(variable.shape), "dimensions": list(variable.dimensions)}
                    for variable in matches
                ]
                if len(matches) != 1:
                    details["warnings"].append(
                        f"Band {band}: expected one array; found {len(matches)}."
                    )

            # Keep each band's time coordinate separate: dates can differ by band.
            for variable in variables:
                if variable.name.lower() not in ("time", "timestamp"):
                    continue
                time_result = {"variable": variable.group().path.rstrip("/") + "/" + variable.name}
                try:
                    time_result.update(inspect_dates(variable))
                except Exception as error:
                    time_result["error"] = str(error)
                    details["warnings"].append(f"{time_result['variable']}: {error}")
                details["times"].append(time_result)
            if not details["times"]:
                details["warnings"].append("No time or timestamp coordinate found.")
        return details

else:
    print("Raw-data inspection and field extraction skipped: rebuild_field_inventory=False.")


### 3.3 Inspect files sequentially

Process one file at a time and save/reuse per-file checkpoints. Errors are recorded without treating failed files as crop absence.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `[<number>/<number>] <filename>: ok or failed` | Progress through the local raw-file list. Successful compatible checkpoints can be reused; failed files are kept explicit. |
| `<crop>: <number> pixels; <number> parcel occurrences` | Counts for this source patch. Parcel occurrences can repeat across patches and are not independent-field counts. |
| `Check: <warning>` | A metadata or data-quality issue to investigate before treating this patch as reliable evidence. |
| `Reason: <message>` | Why a file could not be inspected. A failed inspection must not be counted as evidence that a crop is absent. |
<!-- output-guide:end -->


In [ ]:
if rebuild_field_inventory:
    import csv
    import json
    from datetime import datetime, timezone

    inspection_settings = {
        "crop_codes": crop_codes,
        "bands": required_bands,
        "parcel_background_ids": parcel_background_ids,
        "inspection_version": 1,
    }
    inspection_records = []

    for file_number, local_file in enumerate(dataset_files, start=1):
        filename = local_file.relative_to(dataset_folder).as_posix()
        # Each file gets its own checkpoint, so interruption does not lose earlier work.
        relative_path = local_file.relative_to(raw_year_folder)
        checkpoint_path = inspection_folder / "checkpoints" / relative_path.with_suffix(".nc.json")
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        signature = None
        if local_file.is_file():
            file_info = local_file.stat()
            signature = {"bytes": file_info.st_size, "modified_ns": file_info.st_mtime_ns}

        previous_result = None
        if checkpoint_path.exists() and not repeat_checks:
            try:
                previous_result = json.loads(checkpoint_path.read_text(encoding="utf-8"))
            except (ValueError, OSError):
                previous_result = None

        can_reuse = (
            previous_result is not None
            and signature is not None
            and previous_result.get("signature") == signature
            and previous_result.get("settings") == inspection_settings
            and previous_result.get("status") == "ok"
            and not previous_result.get("warnings")
        )
        if can_reuse:
            result = previous_result
        else:
            result = {
                "filename": filename,
                "signature": signature,
                "settings": inspection_settings,
                "checked_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            try:
                if signature is None:
                    raise FileNotFoundError(
                        "File is missing. Refresh the file list and check Step 2 downloads."
                    )
                result.update(inspect_file(local_file))
            except Exception as error:
                result.update(status="failed", error=f"{type(error).__name__}: {error}")
            temporary_path = checkpoint_path.with_suffix(".json.tmp")
            temporary_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
            temporary_path.replace(checkpoint_path)

        inspection_records.append(result)
        print(f"[{file_number}/{len(dataset_files)}] {Path(filename).name}: {result['status']}")
        if result["status"] == "ok":
            for crop, crop_result in result["crops"].items():
                print(
                    f"  {crop}: {crop_result['pixels']} pixels; {crop_result['parcel_occurrences']} parcel occurrences"
                )
            for warning in result["warnings"]:
                print("  Check:", warning)
        else:
            print("  Reason:", result["error"])

else:
    print("Raw-data inspection and field extraction skipped: rebuild_field_inventory=False.")


### 3.4 Write dataset reports

Use the inspection records from the previous cell to build counts and the Step 4 candidate list.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Reports saved to: <path>` | Location of the crop counts, summary and checkpoints used to select candidate patches. |
| `Candidate files for Step 4: <number>` | Successfully inspected patches containing at least one configured target crop. It counts files, not parcels or accepted seasons. |
| `Parcel occurrences are not unique independent fields.` | A reminder that repeated source records still need geometry and identity review before field-based splitting. |
<!-- output-guide:end -->


In [ ]:
if rebuild_field_inventory:
    # Build a dataset-wide table from all files selected for this run.
    columns = [
        "filename",
        "crop",
        "code",
        "pixels",
        "parcel_occurrences",
        "pixels_without_valid_parcel",
        "missing_bands",
        "warnings",
    ]
    with (inspection_folder / "crop_counts.csv").open("w", newline="", encoding="utf-8") as report:
        writer = csv.DictWriter(report, fieldnames=columns)
        writer.writeheader()
        for result in inspection_records:
            if result["status"] != "ok":
                continue
            missing_bands = [band for band, arrays in result["bands"].items() if not arrays]
            for crop, counts in result["crops"].items():
                writer.writerow(
                    {
                        "filename": result["filename"],
                        "crop": crop,
                        "code": counts["code"],
                        "pixels": counts["pixels"],
                        "parcel_occurrences": counts["parcel_occurrences"],
                        "pixels_without_valid_parcel": counts["pixels_without_valid_parcel"],
                        "missing_bands": "; ".join(missing_bands),
                        "warnings": "; ".join(result["warnings"]),
                    }
                )

    successful_files = [result for result in inspection_records if result["status"] == "ok"]
    candidate_patch_paths = [
        result["filename"]
        for result in successful_files
        if any(counts["pixels"] > 0 for counts in result["crops"].values())
    ]
    crop_inventory_summary = {
        "year": inspection_year,
        "input_folder": str(raw_year_folder),
        "coverage": "all locally downloaded 2020 NetCDF files; remote completeness unverified",
        "settings": inspection_settings,
        "files_selected": len(inspection_records),
        "labels_read_successfully": len(successful_files),
        "failed_files": [
            result["filename"] for result in inspection_records if result["status"] == "failed"
        ],
        "candidate_files": candidate_patch_paths,
        "crops": {},
    }
    for crop in crop_codes:
        counts = [result["crops"][crop] for result in successful_files]
        crop_inventory_summary["crops"][crop] = {
            "pixels": sum(item["pixels"] for item in counts),
            "files_containing_crop": sum(item["pixels"] > 0 for item in counts),
            "known_parcel_occurrences": sum(
                item["parcel_occurrences"]
                for item in counts
                if item["parcel_occurrences"] is not None
            ),
            "files_missing_parcel_counts": sum(
                item["parcel_occurrences"] is None for item in counts
            ),
        }
    (inspection_folder / "summary.json").write_text(
        json.dumps(crop_inventory_summary, indent=2), encoding="utf-8"
    )
    print("Reports saved to:", inspection_folder)
    print("Candidate files for Step 4:", len(candidate_patch_paths))
    print("Parcel occurrences are not unique independent fields.")

else:
    print("Raw-data inspection and field extraction skipped: rebuild_field_inventory=False.")


### Outputs and interpretation

`crop_counts.csv` gives pixel and parcel-occurrence counts. `summary.json` lists coverage, failures and candidates. Per-file JSON checkpoints preserve parcel IDs, band metadata and dates. Zero counts apply only to successfully inspected labels under the chosen encoding.


## 4. Review fields and prepare the balanced pilot

Use Step 3's candidate patches to check image alignment and recover geographic crop fragments. Verify each source coordinate system and pixel grid before extracting polygons. Do not guess a coordinate system when metadata is missing.

Review one patch and date at a time. RGB previews use percentile stretching for display and remain unmasked S4A L1C imagery. They do not independently verify the declared crop.

Keep uncertain field fragments separate. Remove only redundant footprints with matching country, year, crop, parcel ID and geometry. Preserve their source records and report remaining repeated IDs and overlaps.


**Sequential reuse mode:** the raw-file code in this section runs only when `rebuild_field_inventory = True`. With the default `False`, each code cell prints a skip message and continues; no raw NetCDF files are opened.


### 4.2 Load candidates and review settings

Choose one patch/date for display. Boundary extraction later covers all candidate patches.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Candidate patches: <number>` | Source patches loaded from the inspection summary for field review and extraction. No new image interpretation has happened in this settings cell. |
<!-- output-guide:end -->


In [ ]:
if rebuild_field_inventory:
    import json
    import csv
    import hashlib
    import numpy as np
    import matplotlib.pyplot as plt
    from netCDF4 import Dataset, num2date
    from affine import Affine
    from rasterio.features import shapes
    from pyproj import CRS, Transformer
    from shapely.geometry import shape, mapping
    from shapely.ops import unary_union, transform as transform_geometry
    from shapely.strtree import STRtree

    summary_path = inspection_folder / "summary.json"
    inspection_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    candidate_files = inspection_summary["candidate_files"]
    if not candidate_files:
        raise ValueError("Step 3 found no target-crop candidates.")

    field_folder = report_folder.parent / "fields" / "2020"
    field_folder.mkdir(parents=True, exist_ok=True)
    candidate_number = 1  # One-based patch selection for visual review.
    acquisition_number = 1  # One-based date selection; inspect several dates.
    review_status = "unreviewed"  # Use unreviewed, needs_followup or overlay_checked.
    review_note = ""  # Describe alignment, clouds and suspicious labels.
    print("Candidate patches:", len(candidate_files))

else:
    print("Raw-data inspection and field extraction skipped: rebuild_field_inventory=False.")


### 4.3 Define grid and boundary helpers

Validate coordinates and extract target-crop polygons while preserving provenance.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `No printed output` | Defines CRS, pixel-grid and boundary-extraction helpers. Later cells call them to validate and extract geometries. |
<!-- output-guide:end -->


In [ ]:
if rebuild_field_inventory:

    def read_grid(variable):
        """Validate the CRS and pixel-center coordinates before polygonizing."""
        group = variable.group()
        if not hasattr(variable, "crs") or not hasattr(variable, "transform"):
            raise ValueError("Missing source CRS or affine transform.")
        crs_text = str(variable.crs).replace("+init=epsg:", "EPSG:")
        source_crs = CRS.from_user_input(crs_text)
        coefficients = np.asarray(variable.transform).ravel()
        if len(coefficients) != 6:
            raise ValueError("Expected six affine coefficients.")
        grid_transform = Affine(*coefficients)
        if grid_transform.b != 0 or grid_transform.d != 0:
            raise ValueError("Rotated grids need an explicit coordinate validation method.")
        x = np.asarray(group.variables["x"][:])
        y = np.asarray(group.variables["y"][:])
        if tuple(variable.shape[-2:]) != (len(y), len(x)):
            raise ValueError("Coordinate lengths differ from raster shape.")
        expected_x = grid_transform.c + (np.arange(len(x)) + 0.5) * grid_transform.a
        expected_y = grid_transform.f + (np.arange(len(y)) + 0.5) * grid_transform.e
        if not np.allclose(x, expected_x, rtol=0, atol=0.001) or not np.allclose(
            y, expected_y, rtol=0, atol=0.001
        ):
            raise ValueError("Pixel centers disagree with the stored affine transform.")
        return source_crs, grid_transform

    def extract_fragments(local_file, source_filename):
        """Extract target-crop fragments while keeping original provenance."""
        features = []
        with Dataset(local_file, "r") as dataset:
            variables = collect_variables(dataset)
            label_variable = find_one_variable(variables, "labels")
            parcel_variable = find_one_variable(variables, "parcels")
            labels, valid_labels = read_mask(label_variable)
            parcels, valid_parcels = read_mask(parcel_variable)
            label_crs, label_transform = read_grid(label_variable)
            parcel_crs, parcel_transform = read_grid(parcel_variable)
            if (
                labels.shape != parcels.shape
                or label_crs != parcel_crs
                or label_transform != parcel_transform
            ):
                raise ValueError("Label and parcel grids do not match.")
            valid_parcels &= ~np.isin(parcels, parcel_background_ids)
            to_wgs84 = Transformer.from_crs(label_crs, "EPSG:4326", always_xy=True)

            for crop, code in crop_codes.items():
                selected_crop = valid_labels & (labels == code)
                parcel_ids = np.unique(parcels[selected_crop & valid_parcels])
                for parcel_id in parcel_ids:
                    selected = selected_crop & valid_parcels & (parcels == parcel_id)
                    pieces = []
                    for geometry, value in shapes(
                        selected.astype("uint8"), mask=selected, transform=label_transform
                    ):
                        if value == 1:
                            pieces.append(shape(geometry))
                    geometry = unary_union(pieces)
                    if geometry.is_empty or not geometry.is_valid:
                        raise ValueError(f"Invalid geometry for parcel {int(parcel_id)}.")
                    geographic_geometry = transform_geometry(to_wgs84.transform, geometry)
                    if not geographic_geometry.is_valid:
                        raise ValueError("Reprojected geometry is invalid.")
                    west, south, east, north = geographic_geometry.bounds
                    if not (-180 <= west <= east <= 180 and -90 <= south <= north <= 90):
                        raise ValueError("Reprojected coordinates are outside WGS84 bounds.")
                    source_key = f"{source_filename}|{crop}|{int(parcel_id)}"
                    fragment_id = hashlib.sha256(source_key.encode()).hexdigest()[:20]
                    other_labels = (
                        valid_labels & valid_parcels & (parcels == parcel_id) & (labels != code)
                    )
                    touches_edge = bool(
                        selected[0].any()
                        or selected[-1].any()
                        or selected[:, 0].any()
                        or selected[:, -1].any()
                    )
                    properties = {
                        "fragment_id": fragment_id,
                        "field_id": None,
                        "source_file": source_filename,
                        "year": str(getattr(dataset, "patch_year", "unknown")),
                        "country": str(getattr(dataset, "patch_country_code", "unknown")),
                        "tile": str(getattr(dataset, "patch_tile", "unknown")),
                        "crop": crop,
                        "crop_code": code,
                        "parcel_id": int(parcel_id),
                        "crop_pixels": int(selected.sum()),
                        "mixed_label_parcel": bool(other_labels.any()),
                        "touches_patch_edge": touches_edge,
                        "source_crs": label_crs.to_string(),
                        "review_status": "unreviewed",
                        "geometry_status": "candidate_fragment",
                    }
                    features.append(
                        {
                            "type": "Feature",
                            "properties": properties,
                            "geometry": mapping(geographic_geometry),
                        }
                    )
        return features

else:
    print("Raw-data inspection and field extraction skipped: rebuild_field_inventory=False.")


### 4.4 Inspect RGB and outlines

Display one dated image and save the chosen review status/note. Change the selection to inspect other patches or dates.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `RGB image and boundary overlay` | A visual check of image/label alignment for one patch and acquisition. Colors identify configured crop labels; they do not independently prove the crop. |
| `Review notes saved: <path>` | The file storing the selected visual-review status and note. Saving an unreviewed record does not mean it has been approved. |
<!-- output-guide:end -->


In [ ]:
if rebuild_field_inventory:
    # Review one patch/date without loading its full time series.
    if not 1 <= candidate_number <= len(candidate_files):
        raise ValueError("Candidate number is outside the candidate list.")
    selected_filename = candidate_files[candidate_number - 1]
    selected_path = dataset_folder / selected_filename

    with Dataset(selected_path, "r") as dataset:
        variables = collect_variables(dataset)
        label_variable = find_one_variable(variables, "labels")
        labels, valid_labels = read_mask(label_variable)
        parcels, valid_parcels = read_mask(find_one_variable(variables, "parcels"))
        label_crs, label_transform = read_grid(label_variable)
        channels = []
        reference_dates = None
        for band in ["B04", "B03", "B02"]:
            variable = find_one_variable(variables, band)
            band_crs, band_transform = read_grid(variable)
            if (
                variable.shape[-2:] != labels.shape
                or band_crs != label_crs
                or band_transform != label_transform
            ):
                raise ValueError("RGB and label grids differ; explicit resampling is required.")
            time_variable = variable.group().variables["time"]
            dates = list(
                num2date(
                    time_variable[:],
                    time_variable.units,
                    calendar=getattr(time_variable, "calendar", "standard"),
                )
            )
            if reference_dates is not None and dates != reference_dates:
                raise ValueError("RGB acquisition dates differ between bands.")
            reference_dates = dates
            if not 1 <= acquisition_number <= len(dates):
                raise ValueError(f"Acquisition must be between 1 and {len(dates)}.")
            values = np.ma.asarray(variable[acquisition_number - 1], dtype=float).filled(np.nan)
            finite_values = values[np.isfinite(values)]
            if not len(finite_values):
                raise ValueError(f"{band} has no valid pixels for this date.")
            low, high = np.percentile(finite_values, [2, 98])
            channels.append(np.nan_to_num(np.clip((values - low) / max(high - low, 1e-9), 0, 1)))
        rgb = np.stack(channels, axis=-1)
        date_text = str(reference_dates[acquisition_number - 1])

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    for axis in axes:
        axis.imshow(rgb)
        axis.set_axis_off()
    axes[0].set_title("Exploratory RGB (unmasked)")
    colors = {"cabbage": "cyan", "carrot": "magenta", "potato": "orange"}
    for crop, code in crop_codes.items():
        selected = valid_labels & (labels == code)
        if selected.any() and not selected.all():
            axes[1].contour(selected.astype(int), levels=[0.5], colors=colors[crop], linewidths=1)
            axes[1].plot([], [], color=colors[crop], label=crop)
    # Show parcel transitions only where both neighbouring parcel pixels are valid.
    edges = np.zeros(labels.shape, dtype=bool)
    edges[:-1] |= (parcels[:-1] != parcels[1:]) & valid_parcels[:-1] & valid_parcels[1:]
    edges[:, :-1] |= (
        (parcels[:, :-1] != parcels[:, 1:]) & valid_parcels[:, :-1] & valid_parcels[:, 1:]
    )
    edge_overlay = np.zeros((*labels.shape, 4))
    edge_overlay[edges] = [1, 1, 1, 0.6]
    axes[1].imshow(edge_overlay)
    axes[1].legend()
    axes[1].set_title("Target crop outlines and white parcel edges")
    fig.suptitle(f"{Path(selected_filename).name} | {date_text}")
    plt.show()

    # Edit review_status/review_note in settings and rerun this cell to save a review.
    if review_status not in ["unreviewed", "needs_followup", "overlay_checked"]:
        raise ValueError("Unknown review status.")
    review_path = field_folder / "visual_reviews.json"
    reviews = json.loads(review_path.read_text(encoding="utf-8")) if review_path.exists() else {}
    reviews[selected_filename + "|" + date_text] = {
        "source_file": selected_filename,
        "date": date_text,
        "status": review_status,
        "note": review_note,
    }
    review_path.write_text(json.dumps(reviews, indent=2), encoding="utf-8")
    print("Review notes saved:", review_path)

else:
    print("Raw-data inspection and field extraction skipped: rebuild_field_inventory=False.")


### 4.5 Extract all fragments

Checkpoint geographic boundaries and report overlapping or repeated-ID relationships. Do not automatically merge them.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `[<number>/<number>] <filename>: <number> fragments` | Extraction progress and geometric crop fragments recovered from each source patch. A fragment is not automatically an independent field. |
| `Needs investigation: <filename> <message>` | A source patch failed extraction. Its absence from the fragment list is recorded as an error. |
| `Candidate fragments: <number>` | Total recovered fragments before deduplication. These supply candidate geometry for the next stage. |
| `Extraction failures: <number>` | Patches that could not be extracted. Zero means extraction recorded no failures, not that crop labels are independently verified. |
| `Spatial relationships needing review: <number>` | Intersecting or duplicate fragment pairs. This counts relationships, so one fragment can appear in several pairs. |
| `Output folder: <path>` | Where extracted geometries and spatial-review records are saved. |
| `Independent-field IDs remain unset until deduplication is verified.` | Field independence is a separate identity decision and cannot be inferred from the fragment count. |
<!-- output-guide:end -->


In [ ]:
if rebuild_field_inventory:
    # Extract all Step 3 candidates, one file at a time.
    checkpoint_folder = field_folder / "patch_fragments"
    checkpoint_folder.mkdir(exist_ok=True)
    all_features = []
    errors = []
    for number, filename in enumerate(candidate_files, start=1):
        local_file = dataset_folder / filename
        checkpoint_name = hashlib.sha256(filename.encode()).hexdigest() + ".json"
        checkpoint_path = checkpoint_folder / checkpoint_name
        try:
            stat = local_file.stat()
            signature = {
                "file": filename,
                "bytes": stat.st_size,
                "modified_ns": stat.st_mtime_ns,
                "crop_codes": crop_codes,
                "background": parcel_background_ids,
                "version": 1,
            }
            saved = (
                json.loads(checkpoint_path.read_text(encoding="utf-8"))
                if checkpoint_path.exists()
                else None
            )
            if saved and saved.get("signature") == signature:
                features = saved["features"]
            else:
                features = extract_fragments(local_file, filename)
                temporary_path = checkpoint_path.with_suffix(".tmp")
                temporary_path.write_text(
                    json.dumps({"signature": signature, "features": features}),
                    encoding="utf-8",
                )
                temporary_path.replace(checkpoint_path)
            all_features.extend(features)
            print(
                f"[{number}/{len(candidate_files)}] {Path(filename).name}: {len(features)} fragments"
            )
        except Exception as error:
            errors.append({"source_file": filename, "error": f"{type(error).__name__}: {error}"})
            print("Needs investigation:", filename, str(error))

    collection = {"type": "FeatureCollection", "features": all_features}
    (field_folder / "candidate_fragments.geojson").write_text(
        json.dumps(collection), encoding="utf-8"
    )
    (field_folder / "extraction_errors.json").write_text(
        json.dumps(errors, indent=2), encoding="utf-8"
    )

    # Use a spatial index to find intersections without comparing every pair.
    geometries = [shape(feature["geometry"]) for feature in all_features]
    overlap_rows = []
    if geometries:
        spatial_index = STRtree(geometries)
        for first_index, geometry in enumerate(geometries):
            for second_index in spatial_index.query(geometry, predicate="intersects"):
                second_index = int(second_index)
                if second_index <= first_index:
                    continue
                first = all_features[first_index]["properties"]
                second = all_features[second_index]["properties"]
                relation = (
                    "exact_duplicate"
                    if geometry.equals(geometries[second_index])
                    else "overlap_or_boundary_contact"
                )
                overlap_rows.append(
                    {
                        "first_fragment": first["fragment_id"],
                        "second_fragment": second["fragment_id"],
                        "relation": relation,
                        "crop_conflict": first["crop"] != second["crop"],
                    }
                )
    with (field_folder / "spatial_review.csv").open("w", newline="", encoding="utf-8") as report:
        writer = csv.DictWriter(
            report,
            fieldnames=["first_fragment", "second_fragment", "relation", "crop_conflict"],
        )
        writer.writeheader()
        writer.writerows(overlap_rows)

    # Repeated IDs are review hints, not evidence of a shared independent field.
    id_groups = {}
    for feature in all_features:
        properties = feature["properties"]
        key = (properties["country"], properties["year"], properties["parcel_id"])
        id_groups.setdefault(key, []).append(properties["fragment_id"])
    repeated_ids = [
        {"country": key[0], "year": key[1], "parcel_id": key[2], "fragments": ids}
        for key, ids in id_groups.items()
        if len(ids) > 1
    ]
    (field_folder / "repeated_parcel_ids.json").write_text(
        json.dumps(repeated_ids, indent=2), encoding="utf-8"
    )
    print("Candidate fragments:", len(all_features))
    print("Extraction failures:", len(errors))
    print("Spatial relationships needing review:", len(overlap_rows))
    print("Output folder:", field_folder)
    print("Independent-field IDs remain unset until deduplication is verified.")

else:
    print("Raw-data inspection and field extraction skipped: rebuild_field_inventory=False.")


### 4.6 Remove verified redundant footprints

Keep the original file and save a separate retrieval list with duplicate-to-retained links.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `original_fragments: <number>` | Number of geometric fragments entering deduplication. |
| `removed_exact_duplicates: <number>` | Redundant footprints removed only when country, year, crop, parcel ID and geometry agree. Removed-to-retained links are preserved. |
| `download_candidates: <number>` | Retained footprints available for pilot sampling after exact deduplication. |
| `unresolved_repeated_id_groups: <number>` | Parcel-ID groups with different footprints that still require identity review. Repeated IDs alone are not enough to merge them. |
| `remaining_spatial_pairs: <number>` | Remaining intersecting/contacting geometry pairs, not a count of independent fields. |
| `planned_24_bin_composites: <number>` | The annual image count if every retained candidate were retrieved. The later balanced-pilot budget can reduce this plan. |
| `independent_field_count: Not yet verified` | The code has not established how many genuinely independent fields these fragments represent. |
| `Step 5 will use: <path>` | The retained-candidate file used by pilot preparation before the final sampled retrieval list is saved. |
<!-- output-guide:end -->


In [ ]:
if rebuild_field_inventory:
    # Deduplicate before Step 5. Original candidate fragments are preserved.
    # Matching IDs alone are insufficient: geometry and crop must also match.
    from collections import defaultdict
    from shapely.geometry import shape
    from shapely.strtree import STRtree
    import json
    import csv

    original_path = field_folder / "candidate_fragments.geojson"
    original_features = json.loads(original_path.read_text(encoding="utf-8"))["features"]
    original_features.sort(key=lambda feature: feature["properties"]["fragment_id"])
    retained_features = []
    groups = defaultdict(list)
    duplicate_records = []

    for feature in original_features:
        properties = feature["properties"]
        # IDs are scoped by country and year; geometry proves the redundant footprint.
        key = (
            properties["country"],
            properties["year"],
            properties["crop"],
            properties["parcel_id"],
        )
        geometry = shape(feature["geometry"])
        if geometry.is_empty or not geometry.is_valid:
            raise ValueError("Invalid geometry must be resolved before deduplication.")
        matching_feature = None
        for previous in groups[key]:
            if geometry.equals(shape(previous["geometry"])):
                matching_feature = previous
                break
        if matching_feature is None:
            feature["properties"]["duplicate_sources"] = []
            groups[key].append(feature)
            retained_features.append(feature)
        else:
            duplicate_record = {
                "removed_fragment": properties["fragment_id"],
                "retained_fragment": matching_feature["properties"]["fragment_id"],
                "source_file": properties["source_file"],
                "reason": "Same country, year, crop, parcel ID and equal WGS84 geometry",
            }
            duplicate_records.append(duplicate_record)
            matching_feature["properties"]["duplicate_sources"].append(dict(properties))

    # Remaining repeated IDs may be partial fragments. Do not discard or merge them.
    remaining_ids = defaultdict(list)
    for feature in retained_features:
        properties = feature["properties"]
        key = (properties["country"], properties["year"], properties["parcel_id"])
        remaining_ids[key].append(properties)
    unresolved_ids = []
    for key, properties in remaining_ids.items():
        if len(properties) > 1:
            unresolved_ids.append(
                {
                    "country": key[0],
                    "year": key[1],
                    "parcel_id": key[2],
                    "fragment_ids": [item["fragment_id"] for item in properties],
                    "crops": sorted(set(item["crop"] for item in properties)),
                    "status": "Different footprints; verify ID scope and field fragments before merging",
                }
            )

    # Record remaining spatial relationships, including different-ID overlaps.
    geometries = [shape(feature["geometry"]) for feature in retained_features]
    spatial_rows = []
    if geometries:
        tree = STRtree(geometries)
        for first_index, geometry in enumerate(geometries):
            for second_index in tree.query(geometry, predicate="intersects"):
                second_index = int(second_index)
                if second_index <= first_index:
                    continue
                first = retained_features[first_index]["properties"]
                second = retained_features[second_index]["properties"]
                other = geometries[second_index]
                relation = (
                    "equal_geometry"
                    if geometry.equals(other)
                    else "boundary_contact" if geometry.touches(other) else "area_overlap"
                )
                spatial_rows.append(
                    {
                        "first_fragment": first["fragment_id"],
                        "second_fragment": second["fragment_id"],
                        "relation": relation,
                        "same_scoped_parcel_id": (
                            first["country"],
                            first["year"],
                            first["parcel_id"],
                        )
                        == (second["country"], second["year"], second["parcel_id"]),
                        "crop_conflict": first["crop"] != second["crop"],
                    }
                )

    output_path = field_folder / "download_candidates.geojson"
    output_path.write_text(
        json.dumps({"type": "FeatureCollection", "features": retained_features}),
        encoding="utf-8",
    )
    (field_folder / "removed_exact_duplicates.json").write_text(
        json.dumps(duplicate_records, indent=2), encoding="utf-8"
    )
    (field_folder / "unresolved_field_ids.json").write_text(
        json.dumps(unresolved_ids, indent=2), encoding="utf-8"
    )
    with (field_folder / "remaining_spatial_review.csv").open(
        "w", newline="", encoding="utf-8"
    ) as report:
        writer = csv.DictWriter(
            report,
            fieldnames=[
                "first_fragment",
                "second_fragment",
                "relation",
                "same_scoped_parcel_id",
                "crop_conflict",
            ],
        )
        writer.writeheader()
        writer.writerows(spatial_rows)

    deduplication_summary = {
        "original_fragments": len(original_features),
        "removed_exact_duplicates": len(duplicate_records),
        "download_candidates": len(retained_features),
        "unresolved_repeated_id_groups": len(unresolved_ids),
        "remaining_spatial_pairs": len(spatial_rows),
        "planned_24_bin_composites": len(retained_features) * 24,
        "policy": "Exact same-ID/crop/year/country footprints deduplicated; partial fragments and different-ID overlaps retained pending review.",
        "independent_field_count": "Not yet verified",
    }
    (field_folder / "deduplication_summary.json").write_text(
        json.dumps(deduplication_summary, indent=2), encoding="utf-8"
    )
    print(json.dumps(deduplication_summary, indent=2))
    print("Step 5 will use:", output_path)

else:
    print("Raw-data inspection and field extraction skipped: rebuild_field_inventory=False.")


### 4.7 Select 3,400 carrot and 10,000 potato candidate pixels

Prefer 60 geographically spread parcel groups and approximately 50 pixels per group. Add groups when small parcels cannot meet equal quotas, preserving the 100-pixel group cap. These are candidates; the window notebook keeps up to 500 usable unique pixels per crop after quality checks.

**Normal sequential execution resumes here in every mode.** The saved field inventory is loaded independently of skipped raw-data cells.


#### 4.7a Audit unique pixel locations

Recover native pixel centers and exclude conflicting labels/groups before sampling.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Unique unambiguous pixels: {<crop>: <number>}` | Distinct native pixel locations with one consistent crop and parcel group. This is the sampling pool, not the final selected budget. |
| `Ambiguous pixels excluded: <number>` | Locations with conflicting crop/group assignments. They are excluded instead of assigning a guessed label. |
<!-- output-guide:end -->


In [ ]:
# Reset the input folder so rerunning sampling cannot accidentally read its output folder.
field_folder = report_folder.parent / "fields" / "2020"

import math
import json
import hashlib
from collections import defaultdict, Counter
import numpy as np
from affine import Affine
from rasterio.features import rasterize
from pyproj import Transformer
from shapely.geometry import shape, mapping, box
from shapely.ops import transform as transform_geometry, unary_union


def build_pixel_catalog(features):
    """Recover unique label-grid centers and retain all fragment provenance."""
    catalog = {}
    for feature in features:
        properties = feature["properties"]
        source_crs = properties["source_crs"]
        projector = Transformer.from_crs("EPSG:4326", source_crs, always_xy=True)
        geometry = transform_geometry(projector.transform, shape(feature["geometry"]))
        west, south, east, north = geometry.bounds
        # Tolerance prevents floating reprojection noise adding an extra row.
        left = math.floor(west / 10 + 1e-6) * 10
        right = math.ceil(east / 10 - 1e-6) * 10
        bottom = math.floor(south / 10 + 1e-6) * 10
        top = math.ceil(north / 10 - 1e-6) * 10
        width = int(round((right - left) / 10))
        height = int(round((top - bottom) / 10))
        if width < 1 or height < 1 or width * height > 5000000:
            raise ValueError("Unexpected native pixel grid; investigate before sampling.")
        mask = rasterize(
            [(mapping(geometry), 1)],
            out_shape=(height, width),
            transform=Affine(10, 0, left, 0, -10, top),
            dtype="uint8",
        )
        rows, columns = np.where(mask == 1)
        group_key = f"{properties['country']}|{properties['year']}|{properties['parcel_id']}"
        for row, column in zip(rows, columns):
            x = int(round(left + int(column) * 10 + 5))
            y = int(round(top - int(row) * 10 - 5))
            key = (source_crs, x, y)
            record = catalog.setdefault(
                key,
                {
                    "source_crs": source_crs,
                    "x": x,
                    "y": y,
                    "crops": set(),
                    "groups": set(),
                    "source_fragments": set(),
                },
            )
            record["crops"].add(properties["crop"])
            record["groups"].add(group_key)
            record["source_fragments"].add(properties["fragment_id"])
    usable = []
    ambiguous = []
    for record in catalog.values():
        if len(record["crops"]) != 1 or len(record["groups"]) != 1:
            ambiguous.append(
                {
                    **record,
                    "crops": sorted(record["crops"]),
                    "groups": sorted(record["groups"]),
                    "source_fragments": sorted(record["source_fragments"]),
                }
            )
            continue
        usable.append(
            {
                "source_crs": record["source_crs"],
                "x": record["x"],
                "y": record["y"],
                "crop": next(iter(record["crops"])),
                "partition_group": next(iter(record["groups"])),
                "source_fragments": sorted(record["source_fragments"]),
            }
        )
    return (
        sorted(
            usable,
            key=lambda row: (
                row["crop"],
                row["partition_group"],
                row["source_crs"],
                row["x"],
                row["y"],
            ),
        ),
        ambiguous,
    )


def choose_spread_groups(groups, count, seed):
    """Use farthest-point selection for broad geographic coverage."""
    keys = sorted(groups)
    centers = []
    for key in keys:
        rows = groups[key]
        crs = rows[0]["source_crs"]
        if any(row["source_crs"] != crs for row in rows):
            raise ValueError("One parcel group spans coordinate systems; review it first.")
        to_metric = Transformer.from_crs(crs, "EPSG:3857", always_xy=True)
        x, y = to_metric.transform(
            np.mean([row["x"] for row in rows]), np.mean([row["y"] for row in rows])
        )
        centers.append([x, y])
    centers = np.asarray(centers)
    rng = np.random.default_rng(seed)
    selected = [int(rng.integers(len(keys)))]
    distances = np.full(len(keys), np.inf)
    while len(selected) < count:
        newest = selected[-1]
        squared_distance = ((centers - centers[newest]) ** 2).sum(axis=1)
        distances = np.minimum(distances, squared_distance)
        distances[selected] = -1
        selected.append(int(np.argmax(distances)))
    return [keys[index] for index in selected]


candidate_path = field_folder / "download_candidates.geojson"
all_candidates = json.loads(candidate_path.read_text(encoding="utf-8"))["features"]
pixel_catalog, ambiguous_pixels = build_pixel_catalog(all_candidates)
print("Unique unambiguous pixels:", dict(Counter(row["crop"] for row in pixel_catalog)))
print("Ambiguous pixels excluded:", len(ambiguous_pixels))


#### 4.7b Set the expanded pilot budget

Output: `Candidate target per crop: 3000` — pixels selected before cloud and window filtering.

Output: `Usable target per crop: 1000` — final cap after quality filtering.

Output: `Expanded pilot folder: <path>` — separate storage preserving the old pilot.


In [ ]:
from crop_pipeline.sources import sampling as pilot_sampling

candidate_budgets = {crop: bilinear_collection.CANDIDATE_BUDGETS[crop] for crop in ("carrot", "potato")}
preferred_groups_per_crop = 60
max_pixels_per_group = 100
sampling_seed = 42
sampling_resolution = 10
sampling_buffer_metres = 20

# Keep old masks and imagery valid while preparing the larger candidate pool.
field_folder = report_folder.parent / "fields" / "2020" / run_name
pilot_folder = field_folder / "balanced_pilot"
pilot_folder.mkdir(parents=True, exist_ok=True)
print("Candidate target per crop:", candidate_budgets)
print("Usable target per crop:", pilot_sampling.USABLE_PIXELS_PER_CROP)
print("Expanded pilot folder:", field_folder)


#### 4.7c Select groups and pixels

Output: `Selected candidate pixels: <counts>` — exactly 3,000 unique candidate pixels for each crop. Insufficient field capacity raises an error instead of duplicating samples.


In [ ]:
print("Reference capacity:", bilinear_collection.capacity_report(pixel_catalog, candidate_budgets))
# Fail before Earth Engine if genuine field capacity cannot meet the budget.
pilot_regions, sampling_summary = bilinear_collection.prepare_candidates(
    pixel_catalog, candidate_budgets, field_folder, sampling_seed
)
# Read the saved selection rather than sample twice with different ordering.
import csv
with (field_folder / "balanced_pilot" / "selected_pixels.csv").open() as selected_file:
    selected_pixels = list(csv.DictReader(selected_file))
print("Selected candidate pixels:", dict(Counter(row["crop"] for row in selected_pixels)))


#### 4.7d Save footprints and sampling masks

Output: `Download regions: <number>` — selected parcel groups needing imagery.

Output: `Sampling masks: <path>` — masks defining the exact candidate pixels.


In [ ]:
# Masks were saved by prepare_candidates; keep their exact candidate identities.
print("Download regions:", len(pilot_regions))
print("Sampling masks:", pilot_folder / "sampling_masks")


#### 4.7e Inspect the saved pilot

Output: `<summary>` — candidate counts and group counts per crop, before quality filtering.

Output: `Step 5 retrieval list: <path>` — regions consumed by satellite retrieval.


In [ ]:
region_path = field_folder / "sampled_download_regions.geojson"
crop_summary_path = pilot_folder / "sampling_summary.json"
print(json.dumps(sampling_summary, indent=2))
print("Step 5 retrieval list:", region_path)


### Outputs and interpretation

`candidate_fragments.geojson` preserves original fragments. `download_candidates.geojson` removes verified redundant footprints. Review tables record unresolved IDs, overlaps and errors. The pilot adds `sampled_download_regions.geojson`, selected-pixel CSV records and 40 mask GeoTIFFs. Shared field identities and crop validity still need verification.


## 5. Download cloud-screened median composites

Use the pilot's selected-pixel regions, not the full fragment inventory. Each region has 24 calendar half-month intervals in 2020: days 1–15 and days 16–month-end. The prepared 40-region pilot therefore has 960 planned GeoTIFFs.

Apply **SCL and cloud probability before calculating medians**. Exclude SCL invalid pixels, clouds, shadows, cirrus and snow. Also exclude cloud scores at or above 40 and invalid scene edges. Missing cloud-probability matches are reported and excluded. The score cutoff is a development choice, not a percentage of the whole scene covered by clouds. Shadow screening uses SCL; no separate projected-shadow detector is implemented.

First mask each granule. Combine valid granules from the same date in sorted image-index order; the last valid granule wins an overlap. Then calculate the median across distinct dates. This gives each date equal weight. Do not confuse this deterministic overlap rule with cloud-score ranking.

GeoTIFF band order is B3, B4, B5, B8, B11, B2 and `clear_count`. Spectral medians remain unscaled float32 values; missing spectral values are −9999. `clear_count` records supporting dates at each pixel. Empty intervals remain present with count zero. The 10 m grid does not add detail to bands originally measured at 20 m.

Old acquisition downloads remain untouched. Mask and grid settings determine the request folder, so changed settings do not silently reuse incompatible composites. Downloads have no per-run cap and save progress after each file.

References: [L2A collection](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED), [cloud probability](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_CLOUD_PROBABILITY), and [direct-download limits](https://developers.google.com/earth-engine/apidocs/ee-image-getdownloadurl).


### 5.2 Set retrieval and cloud settings

Load pilot regions, dates, output paths and the cloud-score cutoff. This cell does not download imagery.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Candidate fragments: <number>` | Selected pilot retrieval regions loaded from the saved region file. Despite the label, this is the sampled retrieval list, not every original fragment. |
| `Imagery storage: <path>` | Where composite GeoTIFFs are read or saved on this device. |
| `Earth Engine project: <project ID>` | The configured project for later online queries; this message alone does not confirm access. |
| `Observation period: <date> to <date> (end exclusive)` | Imagery-query dates. The current plan is full 2020; cross-year feature windows may require imagery outside this plan. |
<!-- output-guide:end -->


In [ ]:
import os
import json
from pathlib import Path

# Collaborators can set environment variables or edit these two settings.
# dataset_folder was defined in Step 2; its sibling folder holds large imagery.
l2a_folder = Path(
    os.environ.get(
        "L2A_COMPOSITE_FOLDER",
        local_settings.get("imagery_root", str(dataset_folder / "sentinel2_pilot_composites")),
    )
).resolve()
earth_engine_project = os.environ.get(
    "EE_PROJECT",
    local_settings.get("earth_engine_project", "YOUR_EARTH_ENGINE_PROJECT"),
)

collection_name = "COPERNICUS/S2_SR_HARMONIZED"
start_date = "2020-01-01"  # Inclusive.
end_date = "2021-01-01"  # Exclusive; extend if adjacent months are needed.
output_resolution = 10  # Metres; confirmed study grid.
buffer_metres = 20  # Rectangular context around each candidate fragment.
spectral_bands = ["B3", "B4", "B5", "B8", "B11", "B2"]
output_bands = spectral_bands + ["clear_count"]
excluded_scl_classes = [0, 1, 3, 8, 9, 10, 11]
cloud_probability_collection = "COPERNICUS/S2_CLOUD_PROBABILITY"
cloud_probability_threshold = 40  # Exclude scores >= 40; exploratory pilot setting.
nodata_value = -9999  # Used only for missing spectral values; clear_count remains zero.
downloads_per_run = None  # Also set explicitly in the download cell.
authenticate_now = True
refresh_inventory = False

from datetime import date

if date.fromisoformat(start_date) >= date.fromisoformat(end_date):
    raise ValueError("end_date must be later than start_date.")
if output_resolution != 10 or buffer_metres < 0:
    raise ValueError("This implementation uses a 10 m grid and nonnegative buffer.")
if downloads_per_run is not None and downloads_per_run < 1:
    raise ValueError("downloads_per_run must be positive or None.")

l2a_folder = l2a_folder / run_name

fragment_path = (
    report_folder.parent / "fields" / "2020" / run_name / "sampled_download_regions.geojson"
)
if not fragment_path.exists():
    raise FileNotFoundError("Run Step 4 balanced-pilot preparation before creating a Step 5 plan.")
fragments = json.loads(fragment_path.read_text(encoding="utf-8"))["features"]
if not fragments:
    raise ValueError("Step 4 produced no candidate fragments.")
l2a_reports = report_folder.parent / "l2a" / "2020" / run_name
l2a_reports.mkdir(parents=True, exist_ok=True)
print("Candidate fragments:", len(fragments))
print("Imagery storage:", l2a_folder)
print("Earth Engine project:", earth_engine_project)
print("Observation period:", start_date, "to", end_date, "(end exclusive)")

if start_date != "2020-01-01" or end_date != "2021-01-01":
    raise ValueError("This composite plan currently covers full 2020 only.")

if not 0 < cloud_probability_threshold <= 100:
    raise ValueError("Cloud probability threshold must be between 0 (exclusive) and 100.")


### 5.3 Connect to Earth Engine

Use the registered project. Set authenticate_now only when sign-in is needed; never store tokens in the notebook.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Connected to Earth Engine.` | Initialization succeeded for the configured project. No imagery has been retrieved by this cell. |
| `An initialization error` | Authentication, project registration or network access needs attention before online retrieval. Local quality/feature work can use already saved data. |
<!-- output-guide:end -->


### Configure trusted HTTPS certificates

Run this cell before Earth Engine authentication. It supplies the installed `certifi` CA bundle when Python has no explicit certificate-file setting, while preserving custom certificate settings. HTTPS certificate verification remains enabled.

Output: `HTTPS certificate bundle: <path>` — the trusted certificate file used by Python HTTPS connections.

Output: `Trusted CA certificates: <number>` — certificates loaded locally; this does not test a server connection or download imagery.


In [ ]:
import os
import ssl
import certifi

# Python.org installations may lack a default CA bundle; retain custom trust settings.
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
certificate_context = ssl.create_default_context()
print("HTTPS certificate bundle:", os.environ["SSL_CERT_FILE"])
print("Trusted CA certificates:", certificate_context.cert_store_stats()["x509_ca"])


In [ ]:
if retrieve_daily_summaries:
    import ee
    if authenticate_now:
        ee.Authenticate()
    ee.Initialize(project=earth_engine_project)
    print("Connected to Earth Engine.")
else:
    print("Earth Engine connection skipped: retrieval is disabled.")


### 5.4a Load shared satellite retrieval rules

Output: `Shared retrieval year: 2020` — Sen4AgriNet observation year. FranceCrops uses the same helper with 2022.
Output: `Imagery request version: 4` and `Resampling: <description>` — identifies the bilinear source processing needed by the new experiments.


In [ ]:
from crop_pipeline.imagery.retrieval import SatelliteRetriever, REQUEST_VERSION

if retrieve_daily_summaries:
    retriever = SatelliteRetriever(ee, 2020, l2a_folder, l2a_reports, cloud_probability_threshold)
print("Imagery request version:", REQUEST_VERSION)
print("Resampling: B5/B11 bilinear before compositing; quality masks nearest-neighbor")


### 5.4b Reuse the shared acquisition mask

Output: `Masking rules: <description>` — daily summaries and composites use the same screening.


In [ ]:
print("Masking rules: SCL, exact cloud-probability matches and valid scene edges")


### 5.4c Prepare calendar intervals

Output: `Calendar half-month intervals: 24` — all annual bins, including empty intervals.


In [ ]:
from crop_pipeline.imagery.quality import interval_bins

half_month_bins = interval_bins(2020)
print("Calendar half-month intervals:", len(half_month_bins))


### 5.5 Inventory imagery and create the composite plan

Output: `Fragments: <number>` — successfully inventoried regions.

Output: `Composites per fragment: 24` and `Total planned composite files: <number>` — annual retrieval workload.

Output: `Inventory errors: <number>` — regions needing investigation or a retry.


In [ ]:
acquisition_plans, inventory_errors = [], []
if retrieve_daily_summaries:
    acquisition_plans, inventory_errors = retriever.build_plans(fragments, refresh=refresh_inventory)
print("Inventoried fragments:", len(acquisition_plans))
print("Inventory errors:", len(inventory_errors))


### Outputs and interpretation

`composite_plan.json` records intervals and screening rules. `composite_progress/` stores paths, source IDs, support counts and checksums; `missing_cloud_probability.json` records excluded matches. Imagery lives in the configured pilot folder. Only pixels marked 1 in sampling masks belong to the pilot; the extra rectangle pixels are context.


### Daily-summary settings

Run this after the Sentinel-2 plan and connection cells. These small tables support timing detection in the next notebook. They do not download additional GeoTIFFs. Skip retrieval when the saved summaries already exist and match your settings.

The 80% coverage setting must match the quality threshold used in the window notebook.


In [ ]:
import importlib
from crop_pipeline.imagery import quality as crop_workflow

minimum_valid_fraction = 0.8
analysis_folder = report_folder.parent / "analysis" / "2020" / run_name
timeline_folder = analysis_folder / "timelines"


### 5.6 Retrieve observed daily summaries

This cell needs the Step 5 connection, helpers and acquisition plan. Skip it only when using saved observed summaries or explicitly accepting provisional composite-only detection.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `<region ID> observed summaries reused` | Saved actual-date summaries were loaded for this region; that region did not need another retrieval. |
| `<region ID> <number> observed daily summaries` | Actual satellite observation dates summarized for the region. Daily means grouped by acquisition date, not imagery every calendar day. |
| `Observed summary rows: <number>` | Total region/date summary records, including dates with poor or zero coverage. Only eligible observations support timing detection. |
| `Observed summary failed: <region ID> <message>` | This region could not be retrieved; inspect observed_errors.json. Composite fallback does not become actual-date evidence. |
<!-- output-guide:end -->


In [ ]:
observed_rows = []
if retrieve_daily_summaries:
    from crop_pipeline.imagery import timelines as observed_timelines
    observed_rows = observed_timelines.retrieve_observed_timelines(
        acquisition_plans, retriever.mask_acquisition, ee, timeline_folder, minimum_valid_fraction
    )
print("Observed summary rows:", len(observed_rows))


### 5.7 Timing gate: select windows before downloading imagery

Output: `Timing-qualified regions` and `Selected composite intervals` count the regions and eight-bin windows that passed the unchanged automatic policy. They are not usable pixel counts. Reports in `timing_first/` retain rejected outcomes and adjacent-year needs. No thresholds or completeness requirements are relaxed.


In [ ]:
selected_plans = []
if retrieve_daily_summaries:
    selected_plans = bilinear_collection.select_timing_plans(
        acquisition_plans, observed_rows, l2a_reports / "timing_first"
    )
    print("Timing-qualified regions:", len(selected_plans))
    print("Unique sampled pixels planned for download:", bilinear_collection.download_pixel_counts(selected_plans, candidate_budgets))
    print("Selected composite intervals:", sum(len(plan["download_intervals"]) for plan in selected_plans))
else:
    print("Timing gate skipped: daily-summary retrieval is disabled.")


### 5.8 Download timing-qualified composites

Output: `<region> <interval> <status>` — progress for each file.

Output: `Downloaded / Reused / Failed composites: <number>` — new files, verified existing files, and failed requests. Rerun to retry failures.


In [ ]:
collection_ready = False
if download_satellite_imagery and not retrieve_daily_summaries:
    raise ValueError("Enable daily summaries and pass the timing gate before downloading imagery.")
if download_satellite_imagery:
    if not selected_plans:
        raise ValueError("No timing-qualified windows; no imagery will be downloaded.")
    download_pixels = bilinear_collection.require_download_pixel_budget(selected_plans, candidate_budgets)
    print("Verified download-pixel budget:", download_pixels)
    download_summary = retriever.download_composites(selected_plans)
    completed_budget = bilinear_collection.verify_download_completion(
        selected_plans, download_summary, candidate_budgets, l2a_reports / "download_pixel_budget.json"
    )
    collection_ready = True
    print("Downloaded sampled pixels:", completed_budget["downloaded_sampled_pixels"])
else:
    download_summary = {"downloaded": 0, "reused": 0, "failed": 0}
    print("Satellite downloads disabled.")
print("Downloaded composites:", download_summary["downloaded"])
print("Reused composites:", download_summary["reused"])
print("Failed composites:", download_summary["failed"])


### 5.9 Inspect the download report

Output: `Download report: <path>` — full per-file outcomes.

Output: `Failed files remain listed for rerunning: <number>` — unresolved download requests.


In [ ]:
print("Download report:", l2a_reports / "last_composite_run.json")
print("Failed files remain listed for rerunning:", download_summary["failed"])


## Handoff to the window notebook

Keep the sampled region GeoJSON, selected-pixel masks, composite progress records, GeoTIFFs and observed daily summaries together. The next notebook reads these files in a fresh kernel.


## Save the source handoff

Output: `Source handoff: <path>` — paths and observation year used by the four-month notebook. This is an acquisition handoff, not proof that downloads or windows passed quality checks.


In [ ]:
from crop_pipeline.sources.pilot_sources import save_source_manifest, save_json

source_handoff_path = report_folder.parent / "source_handoffs" / f"sen4agrinet_{run_name}.json"
revision_records = {
    str(path.relative_to(report_folder)): json.loads(path.read_text())["revision"]
    for path in sorted((report_folder / "2020").glob("*/download_plan.json"))
}
if collection_ready:
    source_manifest = save_source_manifest(
        source_handoff_path,
        "sen4agrinet",
        2020,
        field_folder,
        l2a_folder,
        l2a_reports / "composite_progress",
        timeline_folder,
        revision={"dataset": dataset_name, "download_plans": revision_records},
    )
    print("Source handoff:", source_handoff_path)
    source_manifest["candidate_target_per_crop"] = candidate_budgets
    save_json(source_handoff_path, source_manifest)
else:
    print("Source handoff not exported: the required download-pixel budget has not completed.")


## Continue to four-month windows

After both source downloads finish, open **03_four_month_windows.ipynb**, set `source_run_name = "bilinear_expanded"` and `allow_existing_version_3 = False`, and run sequentially. The final cap stays at 500 usable pixels per crop. For future feature engineering and training, set `analysis_run_name = "bilinear_expanded"` in those notebooks. Current version 3 pilot outputs remain available in the model-training notebook’s local exploratory mode.
